In [1]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis
from gnomad.utils.vep import process_consequences

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_eval_qc.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe054.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_eval_qc.log


In [3]:
in_wes="/well/ukbb-wes/pvcf/oqfe/ukbb-wes-oqfe-pvcf-chr21.vcf.gz"

In [43]:
# annotate with VEP
mt = qc.get_table(in_wes, 'vcf')
mt = process_consequences(hl.vep(mt, '/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/utils/configs/vep_env.json')) 

2021-10-12 12:07:50 Hail: INFO: Coerced sorted dataset


In [44]:
# explode rows
mt = mt.explode_rows(mt.vep.worst_csq_by_gene_canonical)

In [45]:
# annotate rows
mt = mt.annotate_rows(gene = mt.vep.worst_csq_by_gene_canonical.gene_id)
mt = mt.annotate_rows(csqs = mt.vep.worst_csq_by_gene_canonical.most_severe_consequence)

In [46]:
mt = mt.drop('vep').drop('vep_proc_id').drop('filters')

In [47]:
# Get mean depth per varaint
mt = mt.annotate_rows(mean_dp=hl.agg.mean(mt.DP))
mt = mt.annotate_rows(mean_gq=hl.agg.mean(mt.GQ))

In [49]:
# check if variant is in imputed data
mt.rows()
